In [22]:
import pandas as pd

df=pd.read_excel(r"C:\Users\Lenovo\Desktop\CLASSROOM\DA\projects\customer_segmentation\Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [24]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [25]:
df = df.dropna(subset=['CustomerID'])

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      406829 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 27.9+ MB


In [27]:
df['CustomerID']=df['CustomerID'].astype(int)


In [28]:
df['total_price']=df['UnitPrice']*df['Quantity']

In [29]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,total_price
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


RFM TABLE

In [30]:
import datetime as dt

# reference date (latest date in dataset)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# create RFM table
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'CustomerID': 'count',
    'total_price': 'sum'
})

# rename columns
rfm.columns = ['recency', 'frequency', 'monetary']

rfm.head()

,recency,frequency,monetary
CustomerID,,,
12346,326,2,0.00
12347,2,182,4310.00
12348,75,31,1797.24
12349,19,73,1757.55
12350,310,17,334.40


In [31]:


rfm.head()

,recency,frequency,monetary
CustomerID,,,
12346,326,2,0.00
12347,2,182,4310.00
12348,75,31,1797.24
12349,19,73,1757.55
12350,310,17,334.40


In [32]:
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['f_score'] = pd.qcut(rfm['frequency'], 5, labels=[1,2,3,4,5])
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])

In [33]:
rfm.head()

,recency,frequency,monetary,r_score,f_score,m_score
CustomerID,,,,,,
12346,326,2,0.00,1,1,1
12347,2,182,4310.00,5,5,5
12348,75,31,1797.24,2,3,4
12349,19,73,1757.55,4,4,4
12350,310,17,334.40,1,2,2


In [34]:
# Clean 6-segment RFM assignment using R, F, and M 
rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)

def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    # Champions: bought recently, buy often, spend the most
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    # Loyal: buy often and spend well, maybe not most recent
    elif f >= 4 and m >= 3:
        return 'Loyal Customers'
    # New: bought very recently but low frequency
    elif r >= 4 and f <= 2:
        return 'New Customers'
    # At Risk: used to be good (high F/M) but haven't bought recently
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    # Lost: low on everything
    elif r <= 2 and f <= 2:
        return 'Lost Customers'
    else:
        return 'Others'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

# the real numbers
print(rfm['segment'].value_counts())
print()

# revenue by segment
seg_rev = rfm.groupby('segment')['monetary'].sum().sort_values(ascending=False)
print(seg_rev)
print()

# % of revenue per segment
print((seg_rev / seg_rev.sum() * 100).round(1))

segment
Lost Customers     1070
Others             1070
Champions           946
Loyal Customers     679
New Customers       362
At Risk             245
Name: count, dtype: int64

segment
Champions          5525408.760
Loyal Customers    1245758.911
Others              703032.851
Lost Customers      394565.362
At Risk             238805.970
New Customers       192493.960
Name: monetary, dtype: float64

segment
Champions          66.6
Loyal Customers    15.0
Others              8.5
Lost Customers      4.8
At Risk             2.9
New Customers       2.3
Name: monetary, dtype: float64


In [35]:
#Which segment has most customers?
rfm['segment'].value_counts()

segment
Lost Customers     1070
Others             1070
Champions           946
Loyal Customers     679
New Customers       362
At Risk             245
Name: count, dtype: int64

Revenue is highly concentrated: Champions are 22% of customers but drive 66.6% of revenue.

In [36]:
#Which segment earns most money?
rfm.groupby('segment')['monetary'].sum().sort_values(ascending=False)

segment
Champions          5525408.760
Loyal Customers    1245758.911
Others              703032.851
Lost Customers      394565.362
At Risk             238805.970
New Customers       192493.960
Name: monetary, dtype: float64

In [37]:
#. Average value per segment
rfm.groupby('segment')['monetary'].mean().sort_values(ascending=False)

segment
Champions          5840.812643
Loyal Customers    1834.696482
At Risk             974.718245
Others              657.040048
New Customers       531.751271
Lost Customers      368.752675
Name: monetary, dtype: float64

In [38]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4372 entries, 12346 to 18287
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   recency    4372 non-null   int64  
 1   frequency  4372 non-null   int64  
 2   monetary   4372 non-null   float64
 3   r_score    4372 non-null   int64  
 4   f_score    4372 non-null   int64  
 5   m_score    4372 non-null   int64  
 6   segment    4372 non-null   object 
dtypes: float64(1), int64(5), object(1)
memory usage: 273.2+ KB


In [39]:
rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)

In [40]:
rfm.dtypes

recency        int64
frequency      int64
monetary     float64
r_score        int64
f_score        int64
m_score        int64
segment       object
dtype: object

In [41]:
rfm.head()

,recency,frequency,monetary,r_score,f_score,m_score,segment
CustomerID,,,,,,,
12346,326,2,0.00,1,1,1,Lost Customers
12347,2,182,4310.00,5,5,5,Champions
12348,75,31,1797.24,2,3,4,At Risk
12349,19,73,1757.55,4,4,4,Champions
12350,310,17,334.40,1,2,2,Lost Customers


In [42]:
rfm['segment'].value_counts()

segment
Lost Customers     1070
Others             1070
Champions           946
Loyal Customers     679
New Customers       362
At Risk             245
Name: count, dtype: int64

The number of Lost customers is high fand that need to be improved

In [43]:
rfm.groupby('segment')['monetary'].sum().sort_values(ascending=False)

segment
Champions          5525408.760
Loyal Customers    1245758.911
Others              703032.851
Lost Customers      394565.362
At Risk             238805.970
New Customers       192493.960
Name: monetary, dtype: float64

In [44]:
rfm.groupby('segment')['monetary'].mean().sort_values(ascending=False)

segment
Champions          5840.812643
Loyal Customers    1834.696482
At Risk             974.718245
Others              657.040048
New Customers       531.751271
Lost Customers      368.752675
Name: monetary, dtype: float64

Champions are extremely valuable customers contributing the highest revenue per user.

In [ ]:
rfm.groupby('segment')['frequency'].mean()

In [ ]:
rfm.groupby('segment')['recency'].mean()

In [ ]:
rfm.groupby('segment').agg({
    'monetary': ['count', 'sum', 'mean']
}).sort_values(('monetary', 'sum'), ascending=False)

The analysis revealed that Champions contribute the highest average revenue per customer (~₹9.3K), while Lost Customers contribute the least (~₹368). This indicates strong revenue concentration among high-value segments and highlights opportunities for re-engagement strategies for At-Risk and New customers.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.countplot(data=rfm, x='segment')
plt.xticks(rotation=45)
plt.title("Customers per Segment")

In [ ]:
revenue = rfm.groupby('segment')['monetary'].sum().reset_index()
sns.barplot(data=revenue, x='segment', y='monetary')
plt.xticks(rotation=45)
plt.title("Revenue by Segment")
plt.show()

Although Lost Customers are high in number, they contribute very little revenue, whereas Champions and Loyal Customers are the most contributers.

In [ ]:
rfm.to_csv(r"C:\Users\Lenovo\Desktop\CLASSROOM\DA\projects\customer_segmentation\rfm_final.csv",index=False)

In [ ]:
rfm=rfm.reset_index()

In [ ]:
rfm.info()

## Key Findings
- **Champions**: 946 customers (22% of base) drive **66.6% of revenue**
- Revenue is highly concentrated — retaining Champions is the primary lever
- Lost + Others make up ~49% of customers but only ~13% of revenue
- 